# Volume of Mixing vs Pressure

Computes ΔV_mix(P*) = V_mixed − V_pure_solvent − V_pure_polymer
for 10 pressures from P*=0.8 to 2.0.

Reads volume data produced by the pressure_sweep.sh pipeline:
- **Mixed**:  from  (lx·ly·lz)
- **Pure solvent**:  (block-averaged volumes)
- **Pure polymer**:  (block-averaged volumes)

Manifest files in  point to each run directory.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
import glob
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

plt.rcParams.update({
    'font.family':        'CMU Serif',
    'mathtext.fontset':   'cm',
    'mathtext.rm':        'CMU Serif',
    'font.size':          20,
    'axes.titlesize':     22,
    'axes.labelsize':     25,
    'xtick.labelsize':    23,
    'ytick.labelsize':    23,
    'legend.fontsize':    23,
    'figure.titlesize':   22,
    'axes.unicode_minus': False,
    'figure.dpi':         120,
})

# --- Configuration ---
BASE_DATANAME  = "slab_support_5beads_tall_rho04"
INTERACTION    = "1.0_1.0"
PURE_INTER     = "1.0_0.0"
SLAB_STEPS     = 600000
PURE_STEPS     = 100000

DATA_DIR = Path("../../flow_data_local/volmix_sweep")

PRESSURES = [round(1.0 + i * 0.1, 1) for i in range(11)]
print(f"Pressures: {PRESSURES}")
print(f"Data root: {DATA_DIR.resolve()}")


In [ ]:
# === Sync volume data from Expanse ===
import paramiko, getpass, stat
from pathlib import Path

EXPANSE_HOST = "login.expanse.sdsc.edu"
EXPANSE_USER = "dpollard"
STAGE_DIR    = "/home/dpollard/Documents/lammps_runs/volmix_sweep/volmix_stage"

stage_script = (
    "SWEEP=~/Documents/lammps_runs/volmix_sweep\n"
    "STAGE=${SWEEP}/volmix_stage\n"
    "mkdir -p \"$STAGE\"\n"
    "for P in 1.0 1.1 1.2 1.3 1.4 1.5 1.6 1.7 1.8 1.9 2.0; do\n"
    "  mkdir -p \"$STAGE/p${P}\"\n"
    "  SLAB=$(ls -dt \"$SWEEP\"/slab_*pstar${P}_* 2>/dev/null | head -1)\n"
    "  SOL=$(ls  -dt \"$SWEEP\"/solvent_*pstar${P}_*       2>/dev/null | head -1)\n"
    "  POL=$(ls  -dt \"$SWEEP\"/polymer_*pstar${P}_*       2>/dev/null | head -1)\n"
    "  [ -n \"$SLAB\" ] && cp \"$SLAB\"/output_files/volume_data/box_dimensions_*.dat \"$STAGE/p${P}/\" 2>/dev/null || true\n"
    "  [ -n \"$SOL\"  ] && cp \"$SOL\"/output_files/volume_data/box_dimensions_*.dat  \"$STAGE/p${P}/\" 2>/dev/null || true\n"
    "  [ -n \"$POL\"  ] && cp \"$POL\"/output_files/volume_data/box_dimensions_*.dat  \"$STAGE/p${P}/\" 2>/dev/null || true\n"
    "  cnt=$(ls \"$STAGE/p${P}/\" 2>/dev/null | wc -l)\n"
    "  echo \"  P=${P}: $cnt files staged\"\n"
    "done\n"
)

password = getpass.getpass(f"Expanse password for {EXPANSE_USER}: ")
totp     = getpass.getpass("TOTP / verification code: ")

def auth_handler(title, instructions, prompt_list):
    responses = []
    for prompt, echo in prompt_list:
        if "password" in prompt.strip().lower():
            responses.append(password)
        else:
            responses.append(totp)
    return responses

print("Connecting to Expanse...")
transport = paramiko.Transport((EXPANSE_HOST, 22))
transport.connect()
transport.auth_interactive(EXPANSE_USER, auth_handler)

ssh = paramiko.SSHClient()
ssh._transport = transport

print("Step 1 — staging files on Expanse...")
_, stdout, stderr = ssh.exec_command("bash -s", get_pty=False)
stdout.channel.sendall(stage_script.encode())
stdout.channel.shutdown_write()
print(stdout.read().decode())

print("Step 2 — downloading via SFTP (skips files already present)...")
sftp = ssh.open_sftp()
DATA_DIR.mkdir(parents=True, exist_ok=True)

def sftp_download_dir(sftp, remote_dir, local_dir):
    local_dir = Path(local_dir)
    local_dir.mkdir(parents=True, exist_ok=True)
    for entry in sftp.listdir_attr(remote_dir):
        remote_path = f"{remote_dir}/{entry.filename}"
        local_path  = local_dir / entry.filename
        if stat.S_ISDIR(entry.st_mode):
            sftp_download_dir(sftp, remote_path, local_path)
        else:
            if local_path.exists() and local_path.stat().st_size == entry.st_size:
                continue  # already up to date
            sftp.get(remote_path, str(local_path))

sftp_download_dir(sftp, STAGE_DIR, DATA_DIR)
sftp.close()
ssh.close()
print("Sync complete.")


## Volume parsing functions

In [ ]:
def avg_box_volume(path, skip_frac=0.5):
    """
    Parse box_dimensions_*.dat (columns: step lx ly lz).
    Returns time-averaged volume = mean(lx*ly*lz) over the last (1-skip_frac) fraction.
    """
    data = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) >= 4:
                try:
                    step, lx, ly, lz = float(parts[0]), float(parts[1]), float(parts[2]), float(parts[3])
                    data.append(lx * ly * lz)
                except ValueError:
                    continue
    if not data:
        raise ValueError(f"No data found in {path}")
    data = np.array(data)
    n_skip = int(len(data) * skip_frac)
    return np.mean(data[n_skip:])


def avg_pure_volume(path, skip_frac=0.0):
    """
    Parse vol_pure_*.dat (columns: step press_mean vol_mean rho_mean, block averages).
    Returns mean of vol_mean column over all blocks (skip_frac=0 since runs are short).
    """
    data = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) >= 3:
                try:
                    vol = float(parts[2])  # vol_mean column
                    data.append(vol)
                except ValueError:
                    continue
    if not data:
        raise ValueError(f"No data found in {path}")
    data = np.array(data)
    n_skip = int(len(data) * skip_frac)
    return np.mean(data[n_skip:])


## Load volume data for each pressure

Uses the sweep manifest files written by pressure_sweep.sh to locate each run directory.

In [ ]:
results = []
missing = []

for P in PRESSURES:
    pstr = f"{P:.1f}"
    dataname     = f"{BASE_DATANAME}_pstar{pstr}"
    sol_dataname = f"final_config_{dataname}_{INTERACTION}_{SLAB_STEPS}_solvent_only"
    pol_dataname = f"final_config_{dataname}_{INTERACTION}_{SLAB_STEPS}_polymer_only"

    p_dir = DATA_DIR / f"p{pstr}"

    # Mixed system: box_dimensions from slab run (last 50% of run)
    slab_vol_file = p_dir / f"box_dimensions_{dataname}_{INTERACTION}_{SLAB_STEPS}.dat"

    # Pure solvent volume file
    sol_vol_file  = p_dir / f"box_dimensions_{sol_dataname}_{PURE_INTER}_{PURE_STEPS}.dat"

    # Pure polymer volume file
    pol_vol_file  = p_dir / f"box_dimensions_{pol_dataname}_{PURE_INTER}_{PURE_STEPS}.dat"

    if not all(f.exists() for f in [slab_vol_file, sol_vol_file, pol_vol_file]):
        missing.append(pstr)
        for label, fpath in [("slab", slab_vol_file), ("solvent", sol_vol_file), ("polymer", pol_vol_file)]:
            if not fpath.exists():
                print(f"[SKIP] P*={pstr}: missing {label} → {fpath}")
        continue

    try:
        V_mix = avg_box_volume(slab_vol_file, skip_frac=0.5)
        V_sol = avg_box_volume(sol_vol_file,  skip_frac=0.5)
        V_pol = avg_box_volume(pol_vol_file,  skip_frac=0.5)
        dV    = V_mix - V_sol - V_pol
        results.append({
            "P": P, "V_mix": V_mix, "V_sol": V_sol,
            "V_pol": V_pol, "dV_mix": dV
        })
        print(f"P*={pstr}:  V_mix={V_mix:.2f}  V_sol={V_sol:.2f}  V_pol={V_pol:.2f}  ΔV={dV:+.3f}")
    except Exception as e:
        print(f"[ERROR] P*={pstr}: {e}")
        missing.append(pstr)

df = pd.DataFrame(results)
print(f"\nLoaded {len(df)}/{len(PRESSURES)} pressure points")
if missing:
    print(f"Missing: {missing}")


## Plot ΔV_mix vs P*

In [ ]:
if df.empty:
    print("No data to plot — run pressure_sweep.sh and wait for all jobs to complete.")
else:
    P      = df["P"].values
    V_mix  = df["V_mix"].values
    V_sol  = df["V_sol"].values
    V_pol  = df["V_pol"].values
    dV_mix = df["dV_mix"].values
    V_ref  = V_sol + V_pol        # pure-component total (normalization reference)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # --- Left: ΔV_mix / V_ref  (fractional volume of mixing) ---
    ax = axes[0]
    ax.plot(P, dV_mix / V_ref, "o-", color="steelblue", lw=2, ms=7)
    ax.axhline(0, color="gray", lw=1, ls="--")
    ax.set_xlabel(r"$P^*$")
    ax.set_ylabel(r"$\Delta V_{\rm mix}\;/\;(V_{\rm sol}+V_{\rm pol})$")
    ax.set_title("Volume of Mixing")
    ax.grid(True, alpha=0.3)

    # --- Right: component volumes / V_mix  (volume fractions) ---
    ax2 = axes[1]
    ax2.plot(P, V_sol / V_mix * 100, "s--", label=r"$V_{\rm solvent}/V_{\rm mix}$", color="tomato",   lw=1.5, ms=5)
    ax2.plot(P, V_pol / V_mix * 100, "^--", label=r"$V_{\rm polymer}/V_{\rm mix}$", color="seagreen",  lw=1.5, ms=5)
    ax2.set_xlabel(r"$P^*$")
    ax2.set_ylabel(r"$V_i\;/\;V_{\rm mix}\;[\%]$")
    ax2.set_title("Component Volume Fractions")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    PLOT_DIR = Path("../../flow_data_local/plots/volmix")
    PLOT_DIR.mkdir(parents=True, exist_ok=True)
    plt.savefig(PLOT_DIR / "volume_of_mixing.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {PLOT_DIR / 'volume_of_mixing.png'}")


## Summary table

In [ ]:
if not df.empty:
    from IPython.display import display
    display_df = df.copy()
    display_df.columns = ["P*", "V_mix [σ³]", "V_solvent [σ³]", "V_polymer [σ³]", "ΔV_mix [σ³]"]
    display_df = display_df.round(3)
    display(display_df)
